# EDA - LH vs RH (Local CPU/GPU)

This notebook explores encoding chaining data for healthy participants and scenarios 1 vs 2.
Outputs are saved under results_local_cpu_gpu/eda.
Default mode is smoke for quick validation.

In [2]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu, pointbiserialr

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

BASE_DIR = Path("/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/classifier_notebook")
ENCODING_CSV = Path("/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/dataset/encoding chaining.csv")
OUTPUT_DIR = BASE_DIR / "results_local_cpu_gpu" / "eda"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = "smoke"  # change to "full" for full execution
SMOKE_MAX_ROWS = 60000

TARGET_TASKS = ["Thinking", "Acting"]
TARGET_SCENARIOS = [1, 2]
MOTOR_CHANNELS = ["C3", "Cz", "C4", "FC3", "FC4"]
TARGET_SUBBANDS = ["Alpha", "Beta", "Gamma"]

def savefig(name: str):
    plt.tight_layout()
    path = OUTPUT_DIR / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()

def chain_ratio_from_sequence(seq):
    bits = [c for c in str(seq) if c in "01"]
    if not bits:
        return np.nan
    return bits.count("1") / len(bits)

df = pd.read_csv(ENCODING_CSV)
if RUN_MODE == "smoke" and len(df) > SMOKE_MAX_ROWS:
    df = df.sample(SMOKE_MAX_ROWS, random_state=42).reset_index(drop=True)

if "scenario_id" in df.columns:
    df["scenario_id"] = pd.to_numeric(df["scenario_id"], errors="coerce")

print(f"Data shape: {df.shape}")
print("\nDtypes:")
print(df.dtypes)
display(df.head(3))

inventory_cols = ["subject_id", "scenario", "task", "channel", "subband"]
for col in inventory_cols:
    if col in df.columns:
        print(f"Unique {col}:", df[col].nunique())
        print(df[col].value_counts(dropna=False).head(15))
        print("-" * 70)

if "subject_id" in df.columns:
    als_count = df["subject_id"].astype(str).str.startswith("ALS").sum()
    healthy_count = df["subject_id"].astype(str).str.startswith("id").sum()
    print(f"ALS rows: {als_count}, Healthy rows: {healthy_count}")

rows_before = len(df)
filtered = df.copy()
if "subject_id" in filtered.columns:
    filtered = filtered[filtered["subject_id"].astype(str).str.startswith("id")]
if "scenario_id" in filtered.columns:
    filtered = filtered[filtered["scenario_id"].isin(TARGET_SCENARIOS)]
if "task" in filtered.columns:
    filtered = filtered[filtered["task"].isin(TARGET_TASKS)]

filtered = filtered.copy()
filtered["label"] = np.where(filtered["scenario_id"] == 1, 0, 1)
filtered["label_name"] = filtered["label"].map({0: "LH", 1: "RH"})

if "chain_ratio" not in filtered.columns:
    filtered["chain_ratio"] = np.nan
filtered["chain_ratio"] = pd.to_numeric(filtered["chain_ratio"], errors="coerce")
missing_ratio = filtered["chain_ratio"].isna()
if "chain_sequence" in filtered.columns and missing_ratio.any():
    filtered.loc[missing_ratio, "chain_ratio"] = filtered.loc[missing_ratio, "chain_sequence"].apply(chain_ratio_from_sequence)

if "chain_sequence" in filtered.columns:
    filtered["chain_len"] = filtered["chain_sequence"].astype(str).apply(lambda x: len([c for c in x if c in "01"]))
else:
    filtered["chain_len"] = np.nan

print(f"Rows before filter: {rows_before}")
print(f"Rows after filter: {len(filtered)}")

class_counts = filtered["label_name"].value_counts().reindex(["LH", "RH"])
plt.figure(figsize=(8, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette="Set2")
plt.title("Class Balance: LH vs RH")
plt.ylabel("Samples")
for i, v in enumerate(class_counts.values):
    plt.text(i, v, str(int(v)), ha="center", va="bottom")
savefig("class_balance_overall.png")

task_counts = filtered.groupby(["task", "label_name"]).size().reset_index(name="count")
plt.figure(figsize=(9, 4))
sns.barplot(data=task_counts, x="task", y="count", hue="label_name", palette="Set1")
plt.title("Class Balance by Task")
savefig("class_balance_by_task.png")

imbalance_ratio = class_counts.max() / max(class_counts.min(), 1)
print(f"Imbalance ratio (max/min): {imbalance_ratio:.4f}")

plt.figure(figsize=(9, 4))
for label_name, color in [("LH", "tab:blue"), ("RH", "tab:orange")]:
    tmp = filtered.loc[filtered["label_name"] == label_name, "chain_ratio"].dropna()
    if len(tmp) > 1:
        sns.histplot(tmp, kde=True, stat="density", label=label_name, color=color, bins=35, alpha=0.4)
plt.title("chain_ratio Distribution (LH vs RH)")
plt.legend()
savefig("chain_ratio_hist_kde.png")

if "subband" in filtered.columns:
    plt.figure(figsize=(10, 4))
    sns.boxplot(data=filtered, x="subband", y="chain_len")
    plt.title("Chain Sequence Length by Subband")
    savefig("chain_length_by_subband.png")

mean_ratio = filtered.groupby(["label_name", "channel", "subband"], as_index=False)["chain_ratio"].mean()

def plot_heatmap_for_label(label_name, out_name):
    data = mean_ratio[mean_ratio["label_name"] == label_name]
    if data.empty:
        return
    pv = data.pivot(index="channel", columns="subband", values="chain_ratio")
    plt.figure(figsize=(8, 6))
    sns.heatmap(pv, annot=True, cmap="viridis", fmt=".3f")
    plt.title(f"Mean chain_ratio Heatmap - {label_name}")
    savefig(out_name)

plot_heatmap_for_label("LH", "heatmap_mean_chain_ratio_lh.png")
plot_heatmap_for_label("RH", "heatmap_mean_chain_ratio_rh.png")

pv_lh = mean_ratio[mean_ratio["label_name"] == "LH"].pivot(index="channel", columns="subband", values="chain_ratio")
pv_rh = mean_ratio[mean_ratio["label_name"] == "RH"].pivot(index="channel", columns="subband", values="chain_ratio")
if not pv_lh.empty and not pv_rh.empty:
    common_i = pv_lh.index.intersection(pv_rh.index)
    common_c = pv_lh.columns.intersection(pv_rh.columns)
    diff = pv_rh.loc[common_i, common_c] - pv_lh.loc[common_i, common_c]
    plt.figure(figsize=(8, 6))
    sns.heatmap(diff, annot=True, cmap="coolwarm", center=0.0, fmt=".3f")
    plt.title("Difference Heatmap (RH - LH)")
    savefig("heatmap_diff_rh_minus_lh.png")

motor_df = filtered[
    filtered["channel"].isin(MOTOR_CHANNELS)
    & filtered["subband"].isin(TARGET_SUBBANDS)
].copy()

if not motor_df.empty:
    plt.figure(figsize=(11, 4))
    sns.violinplot(data=motor_df, x="channel", y="chain_ratio", hue="label_name", split=False, inner="quartile")
    plt.title("Motor Channels: chain_ratio by Class")
    savefig("violin_motor_channels.png")

    plt.figure(figsize=(9, 4))
    sns.violinplot(data=motor_df, x="subband", y="chain_ratio", hue="label_name", split=False, inner="quartile")
    plt.title("Motor Subbands: chain_ratio by Class")
    savefig("violin_motor_subbands.png")

mw_rows = []
for ch in sorted(motor_df["channel"].dropna().unique()):
    for sb in sorted(motor_df["subband"].dropna().unique()):
        tmp = motor_df[(motor_df["channel"] == ch) & (motor_df["subband"] == sb)]
        lh = tmp.loc[tmp["label"] == 0, "chain_ratio"].dropna().values
        rh = tmp.loc[tmp["label"] == 1, "chain_ratio"].dropna().values
        if len(lh) >= 3 and len(rh) >= 3:
            stat, p = mannwhitneyu(lh, rh, alternative="two-sided")
            mw_rows.append({"channel": ch, "subband": sb, "u_stat": stat, "p_value": p, "significant": p < 0.05})
mw_df = pd.DataFrame(mw_rows).sort_values("p_value") if mw_rows else pd.DataFrame(columns=["channel", "subband", "u_stat", "p_value", "significant"])
mw_df.to_csv(OUTPUT_DIR / "mannwhitney_motor_pairs.csv", index=False)
print("Significant motor pairs (p < 0.05):")
display(mw_df[mw_df.get("significant", False)].head(20))

corr_rows = []
for (ch, sb), tmp in filtered.groupby(["channel", "subband"]):
    sub = tmp[["chain_ratio", "label"]].dropna()
    if sub["label"].nunique() == 2 and len(sub) >= 6:
        r, p = pointbiserialr(sub["label"], sub["chain_ratio"])
        corr_rows.append({"channel": ch, "subband": sb, "correlation": r, "p_value": p})
corr_df = pd.DataFrame(corr_rows).sort_values(by="correlation", key=lambda x: x.abs(), ascending=False) if corr_rows else pd.DataFrame(columns=["channel", "subband", "correlation", "p_value"])
corr_df.to_csv(OUTPUT_DIR / "pointbiserial_channel_subband.csv", index=False)

if not corr_df.empty:
    pv_corr = corr_df.pivot(index="channel", columns="subband", values="correlation")
    plt.figure(figsize=(8, 6))
    sns.heatmap(pv_corr, annot=True, cmap="RdBu_r", center=0.0, fmt=".3f")
    plt.title("Point-Biserial Correlation (chain_ratio vs label)")
    savefig("correlation_heatmap_chain_ratio_label.png")

top10 = corr_df.head(10)
print("Top 10 discriminative (channel, subband) by |correlation|:")
display(top10)

subj = filtered.groupby(["subject_id", "label_name"], as_index=False)["chain_ratio"].mean()
subj_pv = subj.pivot(index="subject_id", columns="label_name", values="chain_ratio")
if {"LH", "RH"}.issubset(subj_pv.columns):
    plt.figure(figsize=(6, 6))
    sns.scatterplot(x=subj_pv["LH"], y=subj_pv["RH"], s=45)
    mn = min(subj_pv[["LH", "RH"]].min())
    mx = max(subj_pv[["LH", "RH"]].max())
    plt.plot([mn, mx], [mn, mx], linestyle="--", color="gray")
    plt.xlabel("Subject Mean LH chain_ratio")
    plt.ylabel("Subject Mean RH chain_ratio")
    plt.title("Subject-level LH vs RH chain_ratio")
    savefig("subject_level_lh_vs_rh_scatter.png")

plt.figure(figsize=(10, 4))
sns.boxplot(data=filtered, x="task", y="chain_ratio", hue="label_name")
plt.title("Task Comparison (Thinking vs Acting) by Class")
savefig("task_comparison_chain_ratio.png")

task_discrim = []
for task, tmp in filtered.groupby("task"):
    sub = tmp[["chain_ratio", "label"]].dropna()
    if sub["label"].nunique() == 2 and len(sub) >= 6:
        r, p = pointbiserialr(sub["label"], sub["chain_ratio"])
        task_discrim.append({"task": task, "point_biserial_r": r, "p_value": p})
task_discrim_df = pd.DataFrame(task_discrim).sort_values(by="point_biserial_r", key=lambda x: x.abs(), ascending=False) if task_discrim else pd.DataFrame(columns=["task", "point_biserial_r", "p_value"])
task_discrim_df.to_csv(OUTPUT_DIR / "task_discriminability.csv", index=False)

summary_items = [
    {"metric": "run_mode", "value": RUN_MODE},
    {"metric": "rows_before_filter", "value": rows_before},
    {"metric": "rows_after_filter", "value": len(filtered)},
    {"metric": "n_subjects_after_filter", "value": filtered["subject_id"].nunique()},
    {"metric": "class_lh_count", "value": int(class_counts.get("LH", 0))},
    {"metric": "class_rh_count", "value": int(class_counts.get("RH", 0))},
    {"metric": "imbalance_ratio", "value": float(imbalance_ratio)},
    {"metric": "significant_motor_pairs", "value": int(mw_df["significant"].sum()) if not mw_df.empty else 0},
    {"metric": "top_abs_corr", "value": float(corr_df["correlation"].abs().max()) if not corr_df.empty else np.nan},
]
summary_df = pd.DataFrame(summary_items)
summary_df.to_csv(OUTPUT_DIR / "eda_summary.csv", index=False)

print("Saved files:")
for p in sorted(OUTPUT_DIR.glob("*")):
    if p.is_file():
        print(f"- {p.name} ({p.stat().st_size / 1024:.1f} KB)")

Data shape: (60000, 10)

Dtypes:
subject_id            str
scenario              str
scenario_id         int64
filename              str
task                  str
channel               str
subband               str
feature               str
chain_sequence     object
chain_ratio       float64
dtype: object


,subject_id,scenario,scenario_id,filename,task,channel,subband,feature,chain_sequence,chain_ratio
0,id76,scenario4,4,EEGET-ALS Dataset/id76/time1/scenario4/EEG.edf,Resting,Cz,High_Beta,peak_frequency,1000100101000100100100010010001000001010101100...,0.3214
1,id77,scenario7,7,EEGET-ALS Dataset/id77/time1/scenario7/EEG.edf,Resting,C4,Gamma,relative_power,1010100001101000101010110101010001010100111101...,0.4472
2,id18,scenario9,9,EEGET-ALS Dataset/id18/time1/scenario9/EEG.edf,Thinking,Cz,High_Beta,std,1010000011011100010010101101001000100111100101...,0.4476


Unique subject_id: 150
subject_id
id3      452
id129    438
id135    435
id125    435
id2      430
id16     429
id10     428
id52     428
id23     427
id19     427
id50     426
id128    426
id24     423
id72     422
id29     421
Name: count, dtype: int64
----------------------------------------------------------------------
Unique scenario: 9
scenario
scenario1    7092
scenario6    7077
scenario4    7064
scenario2    7056
scenario3    7049
scenario7    7025
scenario5    6929
scenario9    5389
scenario8    5319
Name: count, dtype: int64
----------------------------------------------------------------------
Unique task: 4
task
Typing          15885
Thinking        15864
Resting         15812
Think_Acting    12439
Name: count, dtype: int64
----------------------------------------------------------------------
Unique channel: 3
channel
Cz    20190
C3    20018
C4    19792
Name: count, dtype: int64
----------------------------------------------------------------------
Unique subband: 4
subba

,channel,subband,u_stat,p_value,significant


Top 10 discriminative (channel, subband) by |correlation|:


,channel,subband,correlation,p_value
9,Cz,High_Beta,0.066950,0.236828
5,C4,High_Beta,0.050130,0.395855
11,Cz,Mu,-0.030184,0.594143
7,C4,Mu,-0.029999,0.607218
2,C3,Low_Beta,0.029362,0.622191
0,C3,Gamma,0.028628,0.620218
6,C4,Low_Beta,0.026805,0.644334
10,Cz,Low_Beta,-0.024000,0.679367
8,Cz,Gamma,-0.023452,0.695475
3,C3,Mu,-0.021311,0.714084


Saved files:
- chain_length_by_subband.png (56.9 KB)
- chain_ratio_hist_kde.png (65.8 KB)
- class_balance_by_task.png (38.6 KB)
- class_balance_overall.png (38.0 KB)
- correlation_heatmap_chain_ratio_label.png (73.3 KB)
- eda_summary.csv (0.2 KB)
- heatmap_diff_rh_minus_lh.png (72.0 KB)
- heatmap_mean_chain_ratio_lh.png (73.0 KB)
- heatmap_mean_chain_ratio_rh.png (71.0 KB)
- mannwhitney_motor_pairs.csv (0.2 KB)
- pointbiserial_channel_subband.csv (0.6 KB)
- subject_level_lh_vs_rh_scatter.png (105.7 KB)
- task_comparison_chain_ratio.png (48.7 KB)
- task_discriminability.csv (0.1 KB)
- violin_motor_channels.png (125.0 KB)
- violin_motor_subbands.png (88.1 KB)
